In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [2]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"


Agent pid 2059
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-9e91a42
# github.com:22 SSH-2.0-9e91a42
# github.com:22 SSH-2.0-9e91a42
# github.com:22 SSH-2.0-9e91a42
# github.com:22 SSH-2.0-9e91a42
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [3]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel
!pip install rpy2

!python -c "import rpy2.robjects as ro; print(ro.r('R.version.string'))"
# !R -q -e 'install.packages("rugarch", repos="https://cloud.r-project.org")'

#第一種加速方法
# !R -q -e 'options(repos=c(CRAN="https://packagemanager.posit.co/cran/__linux__/jammy/latest")); install.packages("rugarch"); library(rugarch); packageVersion("rugarch")'

#第二種加速方法
!sudo apt-get update -y
!sudo apt-get install -y r-cran-rugarch
!R -q -e 'library(rugarch); packageVersion("rugarch")'

#檢查版本
!R -q -e 'library(rugarch); cat("version:", as.character(packageVersion("rugarch")), "\n"); cat("libpath:", find.package("rugarch"), "\n")'


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras import mixed_precision

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import precision_score, recall_score, f1_score

from functools import cached_property
from typing import Union, Optional, Dict

from scipy.stats import norm, t as tdist
from scipy.stats import chi2


try:
    from arch import arch_model
    HAS_ARCH = True
except Exception:
    HAS_ARCH = False


#### SET GPU
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs:", len(gpus), gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✓ memory growth set")
    except RuntimeError as e:
        print("⚠️ GPU 已初始化，無法再設定 memory growth：", e)


# （建議）再開啟 XLA 與混合精度
tf.config.optimizer.set_jit(True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("是否可用GPU:", tf.test.is_gpu_available())
print("使用中的裝置:", tf.config.list_physical_devices('GPU'))

print("✓ GPU 初始化流程完成")


# 1. 定義檔案路徑
file_paths = {
    # "bonds_day": "./filtered_output/bonds_day_clean_period.csv",
    # "bonds_hour": "./filtered_output/bonds_hour_clean_period.csv",
    # "crypto_day": "./filtered_output/crypto_day_clean_period.csv",
    # "crypto_hour": "./filtered_output/crypto_hour_clean_period.csv",
    # "others_day": "./filtered_output/others_day_clean_period.csv",
    # "others_hour": "./filtered_output/others_hour_clean_period.csv",
    "stock_day":  "./filtered_output/stock_day_fluctuation_aligned.csv",
    # "stock_hour": "./filtered_output/stock_hour_clean_period.csv"
}


def find_date_col(df):
    for col in df.columns:
        if 'date' in col.lower():
            return col
    return df.columns[0]

def read_and_clean(file):
    # ① 正确地读 CSV，不要写 (index=True)
    df = pd.read_csv(file)

    # ② 找到原始时间列名
    date_col = find_date_col(df)

    # ③ 依次尝试各种格式去解析
    parsed = False
    for fmt in [
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%Y-%m-%d %H:%M',
        '%Y/%m/%d %H:%M',
    ]:
        try:
            df[date_col] = pd.to_datetime(
                df[date_col],
                format=fmt,     # 严格匹配
                errors='raise'  # 抛错就切换下一个 fmt
            )
            parsed = True
            break
        except Exception:
            continue

    # ④ 如果上面都没能解析，再宽松一把
    if not parsed:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    # ⑤ 把分钟/秒都砍掉，保留到「小时」粒度
    df[date_col] = df[date_col].dt.floor('h').dt.tz_localize(None)

    # ⑥ 把这一列重命名为 DATE
    df = df.rename(columns={date_col: 'DATE'})

    # （可选）如果你想让 DATE 作为索引：
    # df = df.set_index('DATE')

    return df



raw_dfs = {}
for name, path in file_paths.items():
    raw_dfs[name] = read_and_clean(path)
    # print(raw_dfs[name].columns)

# ===== 資料結構：取代 all_data =====
class AssetGroupLite:
    def __init__(self, name, df):
        self.name = name
        df = df.copy()
        df['DATE'] = pd.to_datetime(df['DATE'])
        self.raw = df.set_index('DATE').sort_index()

    @cached_property
    def _close_cols(self):
        return [c for c in self.raw.columns if c.endswith('_CLOSE')]

    @cached_property
    def _vol_cols(self):
        return [c for c in self.raw.columns if c.endswith('_VOLUME')]

    @cached_property
    def close_ln(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return np.log(self.raw[self._close_cols]).rename(
            columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln')
        )

    @cached_property
    def close_ln_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        logp = np.log(self.raw[self._close_cols])
        return (
            logp.diff()
               .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_ln_ret'))
               .dropna(how='all')
        )

    @cached_property
    def close_arith_ret(self):
        if not self._close_cols:
            return self.raw.iloc[[]]
        return (
            self.raw[self._close_cols].pct_change()
                .rename(columns=lambda x: x.replace('_CLOSE', '_CLOSE_arith_ret'))
                .dropna(how='all')
        )

class DataRepository:
    REQUIRED_INDEX = "DATE"

    def __init__(self, raw_dfs: dict, check_schema: bool = True):
        self.groups = {}
        for name, df in raw_dfs.items():
            if check_schema:
                assert 'DATE' in df.columns, f"{name}: 缺少 DATE 欄"
            self.groups[name] = AssetGroupLite(name, df)

    # 補上 group()，方便外部與內部呼叫
    def group(self, name: str) -> AssetGroupLite:
        if name not in self.groups:
            raise KeyError(f"Group '{name}' 不存在。可用群組：{list(self.groups.keys())}")
        return self.groups[name]

    def series(self, group: str, series_name: str) -> pd.Series:
        g = self.group(group)
        # 加上 raw → 能抓 OHLCV、IS_TRADING 等原始欄
        search_order = ['close_ln_ret', 'close_arith_ret', 'close_ln', 'raw']

        # 1) 直接命中
        for key in search_order:
            tbl = getattr(g, key)
            if series_name in tbl.columns:
                return tbl[series_name]

        # 2) 容錯：只給 base symbol，自動補候選
        base = (series_name
                .replace('_CLOSE', '')
                .replace('_OPEN', '')
                .replace('_HIGH', '')
                .replace('_LOW', '')
                .replace('_VOLUME', '')
                .replace('_IS_TRADING', '')
                .replace('_CLOSE_ln_ret', '')
                .replace('_CLOSE_arith_ret', '')
                .replace('_CLOSE_ln', ''))
        candidates = [
            f'{base}_CLOSE_ln_ret',
            f'{base}_CLOSE_arith_ret',
            f'{base}_CLOSE_ln',
            f'{base}_OPEN',
            f'{base}_HIGH',
            f'{base}_LOW',
            f'{base}_CLOSE',
            f'{base}_VOLUME',
            f'{base}_IS_TRADING'
        ]
        for cand in candidates:
            for key in search_order:
                tbl = getattr(g, key)
                if cand in tbl.columns:
                    return tbl[cand]

        raise KeyError(f"{group}: 找不到 {series_name} 或候選 {candidates}")

    # 取整張表
    def table(self, group: str, table_name: str) -> pd.DataFrame:
        g = self.group(group)
        if not hasattr(g, table_name):
            raise KeyError(f"{group}: 無表 '{table_name}'。可用表：['close_ln_ret','close_arith_ret','close_ln']")
        return getattr(g, table_name)

    # 若要直接拿 raw 的原始價/量欄位（例如 *_CLOSE 或 *_VOLUME）
    def raw_series(self, group: str, raw_col: str) -> pd.Series:
        g = self.group(group)
        if raw_col not in g.raw.columns:
            raise KeyError(f"{group}: raw 中沒有欄位 {raw_col}")
        return g.raw[raw_col]

repo = DataRepository(raw_dfs)
print("repo success")

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


Num GPUs: 1 [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ memory growth set
是否可用GPU: True
使用中的裝置: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✓ GPU 初始化流程完成
repo success


In [5]:
### import R
# ====== One-time setup ======

import rpy2.robjects as ro
from rpy2.robjects import r
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter
from rpy2.robjects import numpy2ri
from rpy2.rinterface import NULL  # ✅ 用這個取代 r("NULL")

# 關閉全域自動轉換（你已做對）
try:
    numpy2ri.deactivate()
except Exception:
    pass
try:
    from rpy2.robjects import pandas2ri
    pandas2ri.deactivate()
except Exception:
    pass


# 確認 rugarch 可用（你已經成功 library(rugarch)）
r('library(rugarch)')

# 在 R 端定義一個 helper：給 y, X(可空), Xfor(可空)，回傳 sigma_in 與 sigma_next
r('''
fit_garchx_once <- function(y, X=NULL, Xfor=NULL, dist="std") {

  # --- 0) 基本檢查 ---
  y <- as.numeric(y)
  if (any(!is.finite(y))) return(NULL)

  k <- 0
  if (!is.null(X)) {
    X <- as.matrix(X)
    if (nrow(X) != length(y)) return(NULL)
    if (any(!is.finite(X))) return(NULL)
    k <- ncol(X)
  }



  # --- 1) Spec ---
  # omega: 常數項
  # alpha1: ARCH 項，昨天的殘差
  # beta1: GARCH 項，昨天的波動
  # vxreg1: 第一個外生變數的係數 (對應您的 VIX)
  # vxreg2: 第二個外生變數的係數 (對應您的 Volume)
  # shape: Student-t 分配的自由度
  spec <- ugarchspec(
    variance.model = list(
      model = "sGARCH",
      garchOrder = c(1,1),
      external.regressors = X
    ),
    mean.model = list(armaOrder=c(0,0), include.mean=FALSE),
    distribution.model = dist
  )

  # --- 2) Fit：錯誤捕捉 + warning 靜音（不直接判死刑） ---
  fit <- tryCatch(
    withCallingHandlers(
      ugarchfit(
        spec=spec, data=y, solver="hybrid",
        fit.control=list(stationarity=1)
      ),
      warning = function(w) {
        # 把 warning 靜音，最後用 convergence 判定是否真的失敗
        invokeRestart("muffleWarning")
      }
    ),
    error = function(e) NULL
  )

  if (is.null(fit)) return(NULL)
  if (convergence(fit) != 0) return(NULL)

  sig_in <- tryCatch(as.numeric(sigma(fit)), error=function(e) NULL)
  if (is.null(sig_in) || any(!is.finite(sig_in))) return(NULL)

  # --- 3) Forecast：外生變數維度防呆 ---
  if (!is.null(X)) {
    if (is.null(Xfor)) {
      Xfor <- matrix(X[nrow(X),], nrow=1)
    } else {
      Xfor <- as.matrix(Xfor)
    }

    # 維度檢查：一定要 1 x k
    if (nrow(Xfor) != 1 || ncol(Xfor) != k) return(NULL)
    if (any(!is.finite(Xfor))) return(NULL)

    fc <- tryCatch(
      ugarchforecast(fit, n.ahead=1,
                     external.forecasts=list(vregfor=Xfor)),
      error = function(e) NULL
    )
    if (is.null(fc)) return(NULL)

  } else {
    fc <- tryCatch(ugarchforecast(fit, n.ahead=1), error=function(e) NULL)
    if (is.null(fc)) return(NULL)
  }

  # --- 2.5) 係數與診斷資訊（新增） ---
  cf <- tryCatch(coef(fit), error=function(e) NULL)
  mc <- tryCatch(fit@fit$matcoef, error=function(e) NULL)  # matrix: Estimate/SE/t/p
  llh <- tryCatch(fit@fit$LLH, error=function(e) NA_real_)

  if (is.null(cf) || any(!is.finite(cf))) return(NULL)

  sig_next <- tryCatch(as.numeric(sigma(fc)[1]), error=function(e) NA_real_)
  if (!is.finite(sig_next)) return(NULL)

  # --- 4) 回傳（新增 coef/matcoef/LLH） ---
  # 你要的 δ1、δ2 在 rugarch 通常會出現在 coef 裡的 vxreg1, vxreg2（外生變數在 variance equation 的係數
  # coef(fit)：係數值（omega/alpha1/beta1/…/vxreg1/vxreg2/shape）
  # fit@fit$matcoef：係數表（estimate / se / t / p）
  # convergence(fit)：收斂碼（你已用它判斷）
  # fit@fit$LLH：log-likelihood（可選但很有用

  return(list(
    sig_in  = sig_in,
    sig_next= sig_next,
    coef    = cf,
    matcoef = mc,
    llh     = llh,
    conv    = convergence(fit)
  ))


}
''')





# ------------------ Features ------------------
def build_feature_df(repo: DataRepository, group: str, symbol: str) -> pd.DataFrame:
    s_open  = repo.raw_series(group, f'{symbol}_OPEN').asfreq('D')
    s_high  = repo.raw_series(group, f'{symbol}_HIGH').asfreq('D')
    s_low   = repo.raw_series(group, f'{symbol}_LOW').asfreq('D')
    s_close = repo.raw_series(group, f'{symbol}_CLOSE').asfreq('D')
    s_vol   = repo.raw_series(group, f'{symbol}_VOLUME').asfreq('D')
    s_flag = repo.raw_series(group, f'{symbol}_IS_TRADING').asfreq('D')
    s_lnrt  = repo.series(group, f'{symbol}_CLOSE_ln_ret').asfreq('D')
    s_ln  = repo.series(group, f'{symbol}_CLOSE_ln').asfreq('D')
    df = pd.concat([
        s_lnrt.rename(f'{symbol}_LN_RET'),
        s_open.rename(f'{symbol}_OPEN'),
        s_high.rename(f'{symbol}_HIGH'),
        s_low.rename(f'{symbol}_LOW'),
        s_close.rename(f'{symbol}_CLOSE'),
        s_vol.rename(f'{symbol}_VOLUME'),
        s_flag.rename(f'{symbol}_IS_TRADING'),
        s_ln.rename(f'{symbol}_CLOSE_LN')
        ], axis=1)
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.dropna(how='any')

############### GARCH-X
def _ewma_fallback(ret_window: pd.Series, y: pd.Series, lam: float = 0.94):
    # """回傳 (log_sigma_series, sigma_next)；log_sigma_series 對齊 ret_window.index"""
    ewma_var = y.pow(2).ewm(alpha=1 - lam, adjust=False).mean()
    log_sigma_series = 0.5 * np.log(np.maximum(ewma_var.values, 1e-12))
    log_sigma_series = (
        pd.Series(log_sigma_series, index=ewma_var.index)
        .reindex(ret_window.index)
        .ffill()
        .bfill()
    )
    sigma_next = np.sqrt(lam * ewma_var.iloc[-1] + (1 - lam) * y.iloc[-1] ** 2)
    return log_sigma_series, float(sigma_next), {"_fallback": 1}


#抓外生變數的相關性
def _safe_add_vxreg_pvalues(res, coef_dict, r):
    # """
    # res: rpy2 的 R ListVector（fit_garchx_once 回傳）
    # coef_dict: 你已經抓到的係數 dict，這裡會就地新增 vxreg1_p / vxreg2_p
    # r: rpy2.robjects.r
    # 保證：不會 raise；抓不到就填 NaN
    # """
    # 預設先填 NaN（保證一定有這兩欄）
    coef_dict["vxreg1_p"] = np.nan
    coef_dict["vxreg2_p"] = np.nan

    try:
        res_names = list(res.names)
        if "matcoef" not in res_names:
            return

        mc = res.rx2("matcoef")
        # matcoef 可能是 R NULL
        if bool(r["is.null"](mc)[0]):
            return

        # 取 row/col names（任何一步失敗就直接 return，不炸）
        try:
            rownames = list(mc.rownames)
            colnames = list(mc.colnames)
        except Exception:
            return

        # 找 p-value 欄位（不同版本可能不同命名）
        p_candidates = ["Pr(>|t|)", "Pr(>|z|)", "p-value", "pvalue", "P(>|t|)", "P(>|z|)"]
        p_col = None
        for cand in p_candidates:
            if cand in colnames:
                p_col = colnames.index(cand)
                break
        if p_col is None:
            return  # 找不到就維持 NaN

        # 轉成 numpy 矩陣
        mc_np = np.array(mc, dtype=float)

        # 抓 vxreg1 / vxreg2（不存在就維持 NaN）
        for pname in ["vxreg1", "vxreg2"]:
            if pname in rownames:
                rix = rownames.index(pname)
                pv = mc_np[rix, p_col]
                coef_dict[f"{pname}_p"] = float(pv) if np.isfinite(pv) else np.nan

    except Exception:
        # 任何例外：吞掉，維持 NaN，不炸
        return


# fit_vol_per_window 參數解釋：
#    exog_window: 視窗內外生矩陣（T×k）
#    exog_next: 預測下一期用的外生（k,）
#    dist: t 或 normal（rugarch 用 std/norm）
#    fallback: 若估計失敗，用 EWMA 或 NaN
#    min_obs: 最少要幾筆才跑得起來
#    scale: 報酬乘 100 再估計（常見做法，改善數值穩定
def fit_vol_per_window(
    ret_window: pd.Series,
    mode: str = "garch",
    exog_window=None,
    exog_next=None,
    dist: str = "t",
    fallback: str = "ewma",
    min_obs: int = 30,
    scale: float = 100.0,
):
    y_use = ret_window.dropna().astype(float)

    # 定義 fallback：估計失敗就退回 EWMA
    def run_fallback():
        if fallback == "nan":
            return (pd.Series(np.nan, index=ret_window.index), float("nan"), {"_fallback": 1})
        lam = 0.94
        ewma_var = y_use.pow(2).ewm(alpha=1 - lam, adjust=False).mean()
        log_sigma = 0.5 * np.log(np.maximum(ewma_var.values, 1e-12))
        log_series = pd.Series(log_sigma, index=ewma_var.index).reindex(ret_window.index).ffill().bfill()
        s_next = np.sqrt(lam * ewma_var.iloc[-1] + (1 - lam) * y_use.iloc[-1] ** 2)

        return log_series, float(s_next), {"_fallback": 1}

    # 資料太少就直接 fallback
    if len(y_use) < min_obs:
        return run_fallback()

    # ---- 外生變數處理 ----
    X_np, Xfor_np = None, None
    # 把外生變數轉成 numpy matrix，並對齊 y 的 index
    # 如果外生是 DataFrame：用 reindex(y_use.index) 讓 X 跟 y 同日期對齊
    # 如果是 array：直接轉 numpy
    # 不管怎樣，保證 X 是 2D matrix（避免 R 端把 vector 當成一維導致報錯）
    if exog_window is not None:
        if isinstance(exog_window, pd.DataFrame):
            X_df = exog_window.reindex(y_use.index)  # 對齊 y 的 index
            X_np = X_df.values.astype(float)
        else:
            X_np = np.asarray(exog_window, dtype=float)
            if X_np.ndim == 1:
                X_np = X_np.reshape(-1, 1)

        if X_np.ndim == 1:
            X_np = X_np.reshape(-1, 1)

        #得外生變數數量 k，並強制要有 exog_next（避免差一期）
        k = X_np.shape[1]

        # ✅ 建議：GARCH-X 時強制要有 exog_next（避免差一期）
        if exog_next is None:
            return run_fallback()
        ex = np.asarray(exog_next, dtype=float).reshape(-1)
        if ex.shape[0] != k:
            return run_fallback()
        Xfor_np = ex.reshape(1, k)

        # ✅ 不要整窗 fallback：對 y / X 同時做有效列 mask
        mask = np.isfinite(y_use.values) & np.isfinite(X_np).all(axis=1)
        y2 = y_use.values[mask]
        X2 = X_np[mask, :]

        if len(y2) < min_obs:
            return run_fallback()

        y_use = pd.Series(y2, index=y_use.index[mask])  # 保留 index 以便輸出對齊
        X_np = X2

        if not np.isfinite(Xfor_np).all():
            return run_fallback()

    # ---- 準備 R 呼叫 ----
    # 從 R 環境抓你定義的 fit_garchx_once（那個回傳 sigma 的 R function）
    # 如果找不到，代表你還沒跑過 R code cell，就直接報錯提醒
    dist_r = "std" if dist.lower().startswith("t") else "norm"
    y_scaled = (y_use.values * scale).astype(float)

    try:
        fit_fun = r["fit_garchx_once"]
    except Exception:
        raise RuntimeError("R 函數 'fit_garchx_once' 未定義，請先執行 R code 區塊。")

    # 只轉輸入：Python -> R
    #把 Python 資料轉成 R 物件（只轉「輸入」）
    with localconverter(default_converter + numpy2ri.converter):
        y_r = numpy2ri.py2rpy(y_scaled)
        X_r = NULL if X_np is None else numpy2ri.py2rpy(X_np)
        Xfor_r = NULL if Xfor_np is None else numpy2ri.py2rpy(Xfor_np)

    # 呼叫：保持回傳為 R 物件（ListVector 或 NULL），避免 OrdDict 問題
    # 用 default_converter 呼叫，確保 res 不會被轉成 Python dict（你一開始的 rx2 報錯就是因為被轉 dict）
    # 如果 R 那邊回 NULL（估計失敗/不收斂），就 fallback
    try:
        with localconverter(default_converter):
            res = fit_fun(y_r, X_r, Xfor_r, dist_r)
    except Exception:
        return run_fallback()

    if bool(r["is.null"](res)[0]):
        return run_fallback()

    # 取回 sigma（res 是 R ListVector，所以 rx2 穩）


    try:
        sig_in = np.array(res.rx2("sig_in"), dtype=float) / scale
        sig_next = float(np.array(res.rx2("sig_next"), dtype=float).reshape(-1)[0] / scale)

        # --- ✅ 新增：取回係數 coef ---
        coef_r = res.rx2("coef")                 # R named vector
        coef_vals = np.array(coef_r, dtype=float)
        coef_names = list(coef_r.names)          # 係數名稱
        coef_dict = dict(zip(coef_names, coef_vals))
        coef_dict["_fallback"] = 0
        #抓外生變數的P
        _safe_add_vxreg_pvalues(res, coef_dict, r)
        # （可選）抓 conv/llh
        conv = int(np.array(res.rx2("conv"), dtype=int).reshape(-1)[0]) if "conv" in list(res.names) else 0
        llh  = float(np.array(res.rx2("llh"), dtype=float).reshape(-1)[0]) if "llh" in list(res.names) else np.nan
        coef_dict["_conv"] = conv
        coef_dict["_llh"] = llh


    except Exception:
        return run_fallback()


    sig_in = np.maximum(sig_in, 1e-12)
    sig_next = float(max(sig_next, 1e-12))

    log_sigma_series = (
        pd.Series(np.log(sig_in), index=y_use.index)
        .reindex(ret_window.index)
        .ffill()
        .bfill()
    )
    return log_sigma_series, sig_next, coef_dict







################## 風險
def compute_var_es_auto(mu, sigma, alpha,
                        nu=None,                 # scalar 或 Series：t 的自由度（= shape）；NaN 代表用常態
                        price_base=None,
                        standardized_t=True,
                        nu_clip=None             # 推薦 (4.5, 30)；None 表示不截尾
                        ):
    # """
    # 自動判斷分布並計算左尾 VaR/ES（報酬層 + 價格層）。

    # 判斷規則：
    # - nu(shape) 有值 -> 使用 Student-t(df=nu)
    # - nu(shape) 為 NaN/None -> 使用 Normal

    # 參數：
    # - mu, sigma: scalar 或 Series
    # - alpha: 例如 0.05 或 0.01
    # - nu: t 自由度，可為 Series（每期不同）；NaN 表示該期用常態
    # - standardized_t: True -> 使用 Var=1 標準化 t（乘 s=sqrt((nu-2)/nu)）
    # - nu_clip: (lo, hi) 截尾 nu，避免 3~99 造成 VaR 大跳（建議先開 (4.5, 30)）
    # """

    if not (0.0 < alpha < 0.5):
        raise ValueError("alpha 應位於 (0, 0.5)。")

    # --- 對齊 index ---
    mu_s  = pd.Series(mu)
    sig_s = pd.Series(sigma).astype(float).clip(lower=0.0)

    idx = mu_s.index.union(sig_s.index)
    mu_s  = mu_s.reindex(idx)
    sig_s = sig_s.reindex(idx)

    nu_s = None
    if nu is not None:
        nu_s = pd.Series(nu).reindex(idx).astype(float)

    price_s = pd.Series(price_base).reindex(idx) if price_base is not None else None

    # 分布判斷：nu 有值 -> t，nu NaN -> normal
    if nu_s is None:
        is_t = pd.Series(False, index=idx)
    else:
        is_t = nu_s.notna()

    is_norm = ~is_t

    VaR_ret = pd.Series(np.nan, index=idx, dtype=float)
    ES_ret  = pd.Series(np.nan, index=idx, dtype=float)

    # ===== 常態 =====
    if is_norm.any():
        z_alpha = norm.ppf(alpha)     # <0
        phi     = norm.pdf(z_alpha)
        ES_std  = -phi / alpha
        VaR_ret.loc[is_norm] = (mu_s + sig_s * z_alpha).loc[is_norm]
        ES_ret.loc[is_norm]  = (mu_s + sig_s * ES_std ).loc[is_norm]

    # ===== t（nu=shape）=====
    if is_t.any():
        nu_use = nu_s.copy()

        # nu 合法性（ES 需要 nu>2）
        invalid = is_t & (nu_use <= 2)
        if invalid.any():
            bad_idx = list(nu_use[invalid].index[:5])
            raise ValueError(f"t 分布計算 ES 需要 nu>2；前幾個不合法 index：{bad_idx}")

        # 截尾（推薦）
        if nu_clip is not None:
            lo, hi = float(nu_clip[0]), float(nu_clip[1])
            nu_use = nu_use.clip(lower=lo, upper=hi)

        # 逐點 df 的分位數與 pdf
        t_alpha = pd.Series(tdist.ppf(alpha, df=nu_use.values), index=idx)   # <0
        f_t     = pd.Series(tdist.pdf(t_alpha.values, df=nu_use.values), index=idx)

        # 標準 t（loc=0,scale=1）的左尾 ES
        ES_std = - ((nu_use + t_alpha**2) / ((nu_use - 1.0) * alpha)) * f_t

        # Var=1 標準化
        if standardized_t:
            s = np.sqrt((nu_use - 2.0) / nu_use)
        else:
            s = 1.0

        VaR_std = s * t_alpha
        ES_std  = s * ES_std

        VaR_ret.loc[is_t] = (mu_s + sig_s * VaR_std).loc[is_t]
        ES_ret.loc[is_t]  = (mu_s + sig_s * ES_std ).loc[is_t]

    out = pd.DataFrame({'VaR_ret': VaR_ret, 'ES_ret': ES_ret}, index=idx)

    if price_s is not None:
        out['VaR_price'] = price_s * np.exp(out['VaR_ret'])
        out['ES_price']  = price_s * np.exp(out['ES_ret'])

    # 回到 mu 的原 index
    return out.reindex(mu_s.index)



# === 0) (可選) 判斷 t 殘差是否已標準化 Var≈1 ===
def check_t_standardization(eps, sigma) -> float:
    # """
    # 回傳標準化殘差 z = eps/sigma 的樣本變異。
    # ≈1 代表 standardized_t=True；若 ≈ ν/(ν-2) 代表未標準化。
    # """
    z = np.asarray(eps) / np.maximum(np.asarray(sigma), 1e-12)
    z = z[np.isfinite(z)]
    return float(np.var(z, ddof=1))



# ------------------ Config ------------------
ASSET_SYMBOL_ES1 = 'ES1'
ASSET_SYMBOL_VIX = 'VIX'
GROUP_DAY   = 'stock_day'
TARGET_START_STR = '2005-06-01'
TARGET_END_STR   = '2025-06-30'

GARCH_WINDOW_DAY = 252


feat_ES1 = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_ES1)
feat_VIX = build_feature_df(repo, GROUP_DAY, ASSET_SYMBOL_VIX)

df_day = feat_ES1.join(feat_VIX, how='left')

# --- 讓索引成為 datetime（很重要） ---
df_day.index = pd.to_datetime(df_day.index, errors='coerce')
df_day = df_day.sort_index()
df_day = df_day.loc[df_day['ES1_IS_TRADING'] == 1]

# print(df_day.head())
# print(df_day.columns)

df_day.to_csv("./filtered_output/df_day.csv", index=True)


PRED_START = pd.to_datetime(TARGET_START_STR)
PRED_END   = pd.to_datetime(TARGET_END_STR)

# 把回測期限制在資料範圍內
data_start = df_day.index.min()
data_end   = df_day.index.max()
if PRED_START < data_start: PRED_START = data_start
if PRED_END   > data_end:   PRED_END   = data_end

# 若 PRED_START 不是可用交易日，推到 >= PRED_START 的第一個交易日
try:
    PRED_START = df_day.index[df_day.index.searchsorted(PRED_START)]
except Exception:
    # 若整段都沒資料，直接報錯
    raise RuntimeError("資料期間與回測期間沒有交集，請調整 TARGET_START/END。")


first_needed = PRED_START - pd.Timedelta(days=GARCH_WINDOW_DAY)
es1_min = df_day.index.min()
ES1_close_series = df_day['ES1_CLOSE']
if es1_min > first_needed:
    raise RuntimeError(f"Insufficient history: need <= {first_needed}, have from {ES1_close_series.index.min()}")



# ------------------ Model ------------------

# 1) 準備母表（確保排序與期間）
# 全歷史的交易日索引（只留 ES1 開市）
dates_all = df_day.index
# print(dates_all)

# 回測區間（仍然要取，決定哪些 t 需要預測）
mask_period = (df_day.index >= PRED_START) & (df_day.index <= PRED_END)
dates_period = df_day.index[mask_period]

# 2) 方便取用的短名
feat_df = df_day  # 與舊程式一致
rows = []
params_rows = []

# 先做全樣本 lag1（日期 s 這列會是 s-1 的值），為了sigmanext
vix_l1_full = feat_df["VIX_CLOSE_LN"].shift(1)            # ln(VIX_{s-1})
vol_l1_full = np.log(feat_df["ES1_VOLUME"]).shift(1)         # ln(Vol_{s-1})


# 3) 走索引的滑窗，不假設日期連續
#   第 i 筆要預測的是 dates[i]（以 t 表示），窗口是 dates[i-WINDOW_DAY : i)（只到 t-1）

for t in dates_period:
    # 1) 找出 t 在「全歷史」中的**位置**（不是在 dates_period 中的 i）
    pos = dates_all.searchsorted(t)   # 或：pos = df_day.index.get_loc(t)
    if pos < GARCH_WINDOW_DAY:
      continue


    garch_idx = np.arange(pos - GARCH_WINDOW_DAY, pos)  # → [t-GARCH,   ..., t-1]

    garch_dates = dates_all.take(garch_idx)

    # 目標日與 t-1
    t_minus_1 = dates_all[pos - 1]   # 直接往前一格 → t-1
    print(f"pos = {dates_all[pos]},  window = {garch_dates[0]} -> {garch_dates[-1]}")

    # print(f"start {t} ")
    # print(f"start {t} | pos {pos} |　t_minus_1 {t_minus_1}")

    # print(f"garch head={list(garch_dates[:5].date)}")
    # print(f"garch tail={list(garch_dates[-5:].date)}\n")
    # print(f"Xw_period head={list(win_dates[:5].date)}")
    # print(f"Xw_period tail={list(win_dates[-5:].date)}\n")


    # --- GARCH（視窗內，只用到 t-1 的 return） ---
    # 你的 feature_names 中若名稱是 ES1_LN_RET 就用它；否則改用你實際欄名
    # garch_dates有300天做完garch後便做zcore存到log_sigma300，再將其取所需時間段出來


    # (B) 再切到這個 window（長度會和 ret_window 一樣）
    vix_l1_win = vix_l1_full.loc[garch_dates]
    vol_l1_win = vol_l1_full.loc[garch_dates]

    # Z-core
    def z_in_window(s: pd.Series):
      mu = s.mean(skipna=True)
      sd = s.std(skipna=True, ddof=0)
      z  = (s - mu) / (sd if sd and sd > 0 else np.nan)
      return z, mu, sd
    vix_z_win, vix_mu, vix_sd = z_in_window(vix_l1_win)
    vol_z_win, vol_mu, vol_sd = z_in_window(vol_l1_win)

    exog_window = pd.concat(
    [vix_z_win.rename("Z_LNVIX_L1"), vol_z_win.rename("Z_LNVOL_L1")], axis=1
    )  # shape = (252, 2)

    #所以我們用 t-1 當期原始值 去做 Z-score（用同一個 window 的 mu/sd）：
    # t-1 當期的 ln(VIX_{t-1}), ln(Vol_{t-1})
    vix_for = float(feat_df.loc[t_minus_1, "VIX_CLOSE_LN"])
    vol_for = float(np.log(feat_df.loc[t_minus_1, "ES1_VOLUME"]))

    # 用 window 的 mu/sd 做 Z（避免 look-ahead）
    vix_next_z = (vix_for - vix_mu) / vix_sd if vix_sd and vix_sd > 0 else np.nan
    vol_next_z = (vol_for - vol_mu) / vol_sd if vol_sd and vol_sd > 0 else np.nan

    exog_next = np.array([vix_next_z, vol_next_z], dtype=float)  # shape=(2,)


    ret_col = 'ES1_LN_RET' if 'ES1_LN_RET' in feat_df.columns else 'LN_RET'
    log_sigma_series, garch_sigma_hat_t, coef_dict = fit_vol_per_window(
    ret_window=feat_df.loc[garch_dates, ret_col],   # 只含 ≤ t-1
    exog_window=exog_window,                        # (252,2)
    exog_next=exog_next,                            # (2,)
    mode="garch",
    dist="t",                                       # 你現在 rugarch 用 std
    fallback="ewma"
    )

    # print(len(log_sigma_series))

    # --- 真實值 ---
    ln_t   = float(feat_df.loc[t,'ES1_CLOSE_LN'])
    p_t   = float(feat_df.loc[t, 'ES1_CLOSE'])
    p_tm1   = float(feat_df.loc[t_minus_1, 'ES1_CLOSE'])
    r_t   = float(feat_df.loc[t, 'ES1_LN_RET'])

    params_rows.append({
        "t": t,
        "sigma_next": garch_sigma_hat_t,
        **coef_dict
    })

    rows.append({
        'DATE'      : t,
        'price_true': p_t,
        'price_base_aligned': p_tm1,
        'ln_true'   : ln_t,
        'ret_true'  : r_t,
        'sigma_next': garch_sigma_hat_t,
        'mu'        : 0.0,
        'shape' : coef_dict.get("shape", np.nan)
    })

#將garch 係數輸出
params_df = pd.DataFrame(params_rows).set_index("t").sort_index()
print("fallback rate =", params_df["_fallback"].mean())
params_df["ab"] = params_df.get("alpha1", np.nan) + params_df.get("beta1", np.nan)
params_df = params_df.rename(columns={
    "vxreg1": "coef_vix",
    "vxreg2": "coef_vol",
    "alpha1": "alpha",
    "beta1": "beta"
})
params_cols = list(params_df.columns)
if "ab" in params_cols and "coef_vol" in params_cols:
    params_cols.remove("ab")
    insert_at = params_cols.index("coef_vol") + 1
    params_cols.insert(insert_at, "ab")
    params_df = params_df[params_cols]
params_df.to_csv("./LSTM_diagnostics/params_by_day.csv", encoding="utf-8-sig")


# # # ------------------ Evaluation ------------------
# ######################################################################################################
# # ##########  風險
# # 先建 df_out
df_out = pd.DataFrame(rows).set_index("DATE").sort_index()

# # 1) 直接看前後幾筆（最常用）
# print(df_out.head(10))
# print(df_out.tail(10))

# # 2) 看欄位型態/缺值概況
# print(df_out.info())
# print(df_out.isna().sum())

# # 3) 快速描述統計（檢查 sigma 量級是否正常）
# print(df_out[["ret_true", "sigma_next"]].describe())

# 參數
STD_T  = True

mu_series    = df_out['mu']                 # μ̂_t
sigma_series = df_out['sigma_next']         # σ̂_t
price_base   = df_out['price_base_aligned'] # P_{t-1}
shape_series = df_out['shape']

# ===== 95% =====
res = compute_var_es_auto(
    mu_series, sigma_series, alpha=0.05,
    nu=shape_series,
    price_base=price_base,
    standardized_t=STD_T,
    nu_clip=(4.5, 30)
)
df_out['VaR_ret_95'] = res['VaR_ret']
df_out['ES_ret_95']  = res['ES_ret']
if 'VaR_price' in res.columns:
    df_out['VaR_price_95'] = res['VaR_price']
if 'ES_price' in res.columns:
    df_out['ES_price_95']  = res['ES_price']

# ===== 99% =====
res = compute_var_es_auto(
    mu_series, sigma_series, alpha=0.01,
    nu=shape_series,
    price_base=price_base,
    standardized_t=STD_T,
    nu_clip=(4.5, 15)     # 99% 上限收緊
)
df_out['VaR_ret_99'] = res['VaR_ret']
df_out['ES_ret_99']  = res['ES_ret']
if 'VaR_price' in res.columns:
    df_out['VaR_price_99'] = res['VaR_price']
if 'ES_price' in res.columns:
    df_out['ES_price_99']  = res['ES_price']

# 報酬層違規
df_out['viol_95'] = (df_out['ret_true'] < df_out['VaR_ret_95']).astype(int)
df_out['viol_99'] = (df_out['ret_true'] < df_out['VaR_ret_99']).astype(int)

viol_rate_95 = float(df_out['viol_95'].mean())
viol_rate_99 = float(df_out['viol_99'].mean())

# 逐日輸出（修正漏引號）
cols_daily = [
    'price_true','price_base_aligned',
    'ret_true','ln_true',
    'sigma_next',   # ← 修正這裡
    'VaR_ret_95','ES_ret_95','viol_95',
    'VaR_ret_99','ES_ret_99','viol_99',
    'VaR_price_95','ES_price_95','VaR_price_99','ES_price_99'
]
df_export = df_out[[c for c in cols_daily if c in df_out.columns]].copy()
print(df_export.head(10))
print(df_export.tail(10))

# 6b) 摘要矩陣（風控報告用）
var_es_matrics = pd.DataFrame({
    'Metric': [
        'Mean_ret_true', 'Std_ret_true',
        'Mean_VaR_ret_95', 'Mean_ES_ret_95',
        'Mean_VaR_ret_99', 'Mean_ES_ret_99',
        'Viol_rate_95', 'Viol_count_95',
        'Viol_rate_99', 'Viol_count_99',
    ],
    'Value': [
        float(df_out['ret_true'].mean()),
        float(df_out['ret_true'].std(ddof=1)),
        float(df_out['VaR_ret_95'].mean()),
        float(df_out['ES_ret_95'].mean()),
        float(df_out['VaR_ret_99'].mean()),
        float(df_out['ES_ret_99'].mean()),
        viol_rate_95, int(df_out['viol_95'].sum()),
        viol_rate_99, int(df_out['viol_99'].sum()),
    ]
})

# === 匯出檔案 ===
output_dir = "./LSTM_diagnostics"
os.makedirs(output_dir, exist_ok=True)
summary_path = os.path.join(output_dir, "var_es_matrics.csv")
var_es_matrics.to_csv(summary_path, index=False)
daily_path = os.path.join(output_dir, "var_es_daily.csv")
df_export.to_csv(daily_path, index=True)  # index=DATE


stem = f"{ASSET_SYMBOL_ES1.lower()}_garch"
fig1 = f"./LSTM_diagnostics/{stem}_VaR_ES_returns_{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"

plt.figure(figsize=(12, 6))

# === 圖 1: 報酬 vs VaR/ES (95%) ===
# True return：淺藍色
plt.plot(
    df_out.index, df_out['ret_true'],
    label='True Return',
    color='blue', alpha=0.7, linewidth=0.8
)

# VaR：綠色曲線（你要的「預測的 VaR 曲線」）
plt.plot(
    df_out.index, df_out['VaR_ret_95'],
    label='VaR 95%',
    color='green', linewidth=0.7, alpha=0.8
)

# # ES：保留（若你也想一起改色可再說）
# plt.plot(
#     df_out.index, df_out['ES_ret_95'],
#     label='ES 95%',
#     color='black', linestyle='--', linewidth=0.8, alpha=0.7
# )

# 違規：紅色「點」
viol = df_out['ret_true'] < df_out['VaR_ret_95']
plt.scatter(
    df_out.index[viol], df_out.loc[viol, 'ret_true'],
    color='red', marker='o', s=14, edgecolors='none', alpha=0.9,
    label='Violation'
)

plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel("Time")
plt.ylabel("Log Return")
plt.title(f"{ASSET_SYMBOL_ES1} Return vs VaR/ES (95%)")
plt.legend()
plt.grid(True)

plt.savefig(fig1, dpi=300, bbox_inches='tight')
plt.close()




串流輸出內容已截斷至最後 5000 行。
pos = 2005-12-30 00:00:00,  window = 2005-01-03 00:00:00 -> 2005-12-29 00:00:00
pos = 2006-01-03 00:00:00,  window = 2005-01-04 00:00:00 -> 2005-12-30 00:00:00
pos = 2006-01-04 00:00:00,  window = 2005-01-05 00:00:00 -> 2006-01-03 00:00:00
pos = 2006-01-05 00:00:00,  window = 2005-01-06 00:00:00 -> 2006-01-04 00:00:00
pos = 2006-01-06 00:00:00,  window = 2005-01-07 00:00:00 -> 2006-01-05 00:00:00
pos = 2006-01-09 00:00:00,  window = 2005-01-10 00:00:00 -> 2006-01-06 00:00:00
pos = 2006-01-10 00:00:00,  window = 2005-01-11 00:00:00 -> 2006-01-09 00:00:00
pos = 2006-01-11 00:00:00,  window = 2005-01-12 00:00:00 -> 2006-01-10 00:00:00
pos = 2006-01-12 00:00:00,  window = 2005-01-13 00:00:00 -> 2006-01-11 00:00:00
pos = 2006-01-13 00:00:00,  window = 2005-01-14 00:00:00 -> 2006-01-12 00:00:00
pos = 2006-01-16 00:00:00,  window = 2005-01-18 00:00:00 -> 2006-01-13 00:00:00
pos = 2006-01-17 00:00:00,  window = 2005-01-19 00:00:00 -> 2006-01-16 00:00:00
pos = 2006-01-18 00

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import chi2
import matplotlib.pyplot as plt
from scipy.stats import norm

def kupiec_test(viol, alpha: float):
    # """
    # Kupiec (POF / Unconditional Coverage) Test
    # H0: P(violation)=alpha  (違規率正確)
    # viol: 0/1 序列（list/np.array/pd.Series）
    # alpha: 左尾機率，例如 0.05 或 0.01
    # 回傳 dict: n, x, viol_rate, LR_uc, p_value
    # """
    v = pd.Series(viol).dropna().astype(int).values
    n = len(v)
    if n == 0:
        raise ValueError("viol 序列為空")

    x = int(v.sum())
    p_hat = x / n

    # 避免 log(0)
    eps = 1e-12
    p_hat_c = np.clip(p_hat, eps, 1 - eps)
    a_c = np.clip(alpha, eps, 1 - eps)

    # log-likelihood under H0 and under MLE
    ll_h0  = (n - x) * np.log(1 - a_c) + x * np.log(a_c)
    ll_mle = (n - x) * np.log(1 - p_hat_c) + x * np.log(p_hat_c)

    LR_uc = -2.0 * (ll_h0 - ll_mle)
    p_val = 1.0 - chi2.cdf(LR_uc, df=1)

    return {
        "test": "Kupiec UC (POF)",
        "alpha": float(alpha),
        "n": int(n),
        "x": int(x),
        "viol_rate": float(p_hat),
        "LR_uc": float(LR_uc),
        "p_value": float(p_val),
    }


def christoffersen_cc_test(viol, alpha: float):
    # """
    # Christoffersen Conditional Coverage (CC) Test
    # H0: (1) 違規率=alpha (UC) AND (2) 違規獨立 (IND)
    # viol: 0/1 序列（list/np.array/pd.Series）
    # alpha: 左尾機率，例如 0.05 或 0.01
    # 回傳 dict: UC/IND/CC 的 LR 統計量與 p-value，以及轉移計數 n00,n01,n10,n11
    # """
    v = pd.Series(viol).dropna().astype(int).values
    n = len(v)
    if n < 2:
        raise ValueError("CC test 需要至少 2 筆資料（才能算轉移）")

    # --- 1) UC（Kupiec） ---
    uc = kupiec_test(v, alpha)
    LR_uc = uc["LR_uc"]

    # --- 2) IND（獨立性：Markov 轉移） ---
    v_lag = v[:-1]
    v_now = v[1:]

    n00 = int(np.sum((v_lag == 0) & (v_now == 0)))
    n01 = int(np.sum((v_lag == 0) & (v_now == 1)))
    n10 = int(np.sum((v_lag == 1) & (v_now == 0)))
    n11 = int(np.sum((v_lag == 1) & (v_now == 1)))

    # 轉移機率（避免除 0）
    eps = 1e-12
    pi01 = n01 / max(n00 + n01, 1)   # P(hit_t=1 | hit_{t-1}=0)
    pi11 = n11 / max(n10 + n11, 1)   # P(hit_t=1 | hit_{t-1}=1)
    pi   = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)  # 無條件（對轉移樣本 n-1）

    pi01_c = np.clip(pi01, eps, 1 - eps)
    pi11_c = np.clip(pi11, eps, 1 - eps)
    pi_c   = np.clip(pi,   eps, 1 - eps)

    # 對數概似：
    # H0(獨立)：同一個 pi
    ll_ind_h0 = (n00 + n10) * np.log(1 - pi_c) + (n01 + n11) * np.log(pi_c)

    # H1(一階 Markov)：兩個轉移機率 pi01, pi11
    ll_ind_h1 = (n00) * np.log(1 - pi01_c) + (n01) * np.log(pi01_c) \
              + (n10) * np.log(1 - pi11_c) + (n11) * np.log(pi11_c)

    LR_ind = -2.0 * (ll_ind_h0 - ll_ind_h1)
    p_ind  = 1.0 - chi2.cdf(LR_ind, df=1)

    # --- 3) CC（條件覆蓋） ---
    LR_cc = LR_uc + LR_ind
    p_cc  = 1.0 - chi2.cdf(LR_cc, df=2)

    return {
        "test": "Christoffersen CC",
        "alpha": float(alpha),
        "n": int(n),
        "x": int(np.sum(v)),
        "viol_rate": float(np.mean(v)),
        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "LR_uc": float(LR_uc),
        "p_uc": float(uc["p_value"]),
        "LR_ind": float(LR_ind),
        "p_ind": float(p_ind),
        "LR_cc": float(LR_cc),
        "p_cc": float(p_cc),
    }


# =======================
# 用 df_export 直接跑
# =======================
# VaR 95%：alpha=0.05
kupiec_95 = kupiec_test(df_export["viol_95"], alpha=0.05)
cc_95     = christoffersen_cc_test(df_export["viol_95"], alpha=0.05)

# VaR 99%：alpha=0.01
kupiec_99 = kupiec_test(df_export["viol_99"], alpha=0.01)
cc_99     = christoffersen_cc_test(df_export["viol_99"], alpha=0.01)


def build_backtest_summary(kupiec_dict, cc_dict, level_tag: str):
    # """
    # level_tag: '95' or '99'
    # """
    alpha = float(kupiec_dict["alpha"])
    n = int(kupiec_dict["n"])
    x = int(kupiec_dict["x"])
    viol_rate = float(kupiec_dict["viol_rate"])
    expected_viol = alpha * n

    return {
        "level": level_tag,
        "alpha": alpha,
        "n": n,
        "x": x,
        "viol_rate": viol_rate,
        "expected_viol": expected_viol,

        "LR_uc": float(kupiec_dict["LR_uc"]),
        "p_uc": float(kupiec_dict["p_value"]),

        "LR_ind": float(cc_dict["LR_ind"]),
        "p_ind": float(cc_dict["p_ind"]),

        "LR_cc": float(cc_dict["LR_cc"]),
        "p_cc": float(cc_dict["p_cc"]),

        "pass_uc_5pct": float(kupiec_dict["p_value"]) > 0.05,
        "pass_cc_5pct": float(cc_dict["p_cc"]) > 0.05,
    }


def build_transition_row(cc_dict, level_tag: str):
    # """
    # 把 n00 n01 n10 n11 與轉移機率 pi01 pi11 存起來，方便看群聚。
    # """
    n00 = int(cc_dict["n00"])
    n01 = int(cc_dict["n01"])
    n10 = int(cc_dict["n10"])
    n11 = int(cc_dict["n11"])

    # 轉移機率（避免除 0）
    denom0 = (n00 + n01)
    denom1 = (n10 + n11)
    denom2 = (n01 + n11)
    pi01 = n01 / denom0 if denom0 > 0 else float("nan")
    pi11 = n11 / denom1 if denom1 > 0 else float("nan")

    #在「今天違約」的日子中，有多大比例「昨天也違約」。
    pi_new = n11 / denom2 if denom2 > 0 else float("nan")

    return {
        "level": level_tag,
        "alpha": float(cc_dict["alpha"]),
        "n": int(cc_dict["n"]),
        "x": int(cc_dict["x"]),
        "viol_rate": float(cc_dict["viol_rate"]),

        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "pi01": pi01,
        "pi11": pi11,
        "pi_new": pi_new,

        "pi11_minus_pi01": (pi11 - pi01) if (pd.notna(pi11) and pd.notna(pi01)) else float("nan"),

        "LR_ind": float(cc_dict["LR_ind"]),
        "p_ind": float(cc_dict["p_ind"]),
    }


# ====== 組表 ======
summary_rows = [
    build_backtest_summary(kupiec_95, cc_95, "95"),
    build_backtest_summary(kupiec_99, cc_99, "99"),
]
df_summary = pd.DataFrame(summary_rows)

transition_rows = [
    build_transition_row(cc_95, "95"),
    build_transition_row(cc_99, "99"),
]
df_transition = pd.DataFrame(transition_rows)

# ====== 存檔 ======
# output_dir = "./LSTM_diagnostics"
# os.makedirs(output_dir, exist_ok=True)

backtest_path = os.path.join(output_dir, "backtest_summary.csv")
trans_path    = os.path.join(output_dir, "transition_matrix.csv")

df_summary.to_csv(backtest_path, index=False, encoding="utf-8-sig")
df_transition.to_csv(trans_path, index=False, encoding="utf-8-sig")

print(f"✅ saved: {backtest_path}")
print(f"✅ saved: {trans_path}")

print("\n--- backtest_summary ---")
print(df_summary)

print("\n--- transition_matrix ---")
print(df_transition)



# =========================
# 1) UC 覆蓋率圖（含 95% CI）
# =========================
def wilson_ci(x: int, n: int, z: float = 1.96):
    # """
    # Wilson score interval for binomial proportion.
    # 回傳 (low, high)
    # """
    if n <= 0:
        return (np.nan, np.nan)
    p = x / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

def plot_uc_coverage(df_export: pd.DataFrame, out_path: str = "uc_coverage.png"):
    # 兩個水準：95% VaR -> alpha=0.05；99% VaR -> alpha=0.01
    specs = [
        ("95%", 0.05, "viol_95"),
        ("99%", 0.01, "viol_99"),
    ]

    rows = []
    for level, alpha, col in specs:
        v = df_export[col].dropna().astype(int).values
        n = len(v)
        x = int(v.sum())
        p_hat = x / n if n > 0 else np.nan
        ci_low, ci_high = wilson_ci(x, n, z=1.96)
        rows.append((level, alpha, n, x, p_hat, ci_low, ci_high))

    res = pd.DataFrame(rows, columns=["level", "alpha", "n", "x", "viol_rate", "ci_low", "ci_high"])

    # 畫圖
    fig, ax = plt.subplots(figsize=(7, 4))
    xs = np.arange(len(res))

    # 點 + 誤差棒（違規率估計與 95% CI）
    y = res["viol_rate"].values
    yerr = np.vstack([y - res["ci_low"].values, res["ci_high"].values - y])
    ax.errorbar(xs, y, yerr=yerr, fmt='o', capsize=4, label="Observed violation rate (95% CI)")

    # 理論線：alpha
    ax.plot(xs, res["alpha"].values, linestyle='--', marker='s', label="Theoretical alpha")

    ax.set_xticks(xs)
    ax.set_xticklabels([f"{lv}\n(n={n}, x={x})" for lv, n, x in zip(res["level"], res["n"], res["x"])])
    ax.set_ylabel("Violation Rate")
    ax.set_title("Unconditional Coverage (UC): Observed vs Theoretical")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    return res  # 若你想順便存成表也方便


# =========================
# 2) Hit timeline（違規時間線）
# =========================
def plot_hit_timeline(df_export: pd.DataFrame, out_path: str = "hit_timeline.png"):
    # """
    # 畫 95% 與 99% 的違規時間點（1 的日期），上下兩條軸並排。
    # """
    if not isinstance(df_export.index, pd.DatetimeIndex):
        # 若 DATE 是欄位而不是 index
        if "DATE" in df_export.columns:
            df_export = df_export.set_index(pd.to_datetime(df_export["DATE"]))
        else:
            raise ValueError("df_export 需要 DatetimeIndex，或包含 DATE 欄位。")

    v95 = df_export["viol_95"].fillna(0).astype(int)
    v99 = df_export["viol_99"].fillna(0).astype(int)

    dates = df_export.index

    fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)

    # 95% hit
    hit_dates_95 = dates[v95.values == 1]
    axes[0].eventplot(hit_dates_95, lineoffsets=1, linelengths=0.8)
    axes[0].set_yticks([1])
    axes[0].set_yticklabels(["VaR 95% hit"])
    axes[0].set_title("Hit Timeline (Violations Over Time)")
    axes[0].grid(True, axis='x', alpha=0.3)

    # 99% hit
    hit_dates_99 = dates[v99.values == 1]
    axes[1].eventplot(hit_dates_99, lineoffsets=1, linelengths=0.8)
    axes[1].set_yticks([1])
    axes[1].set_yticklabels(["VaR 99% hit"])
    axes[1].grid(True, axis='x', alpha=0.3)

    axes[1].set_xlabel("Time")

    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.close(fig)


# =========================
# 直接呼叫（你只要確保 df_export 已存在）
# =========================
uc_table = plot_uc_coverage(df_export, out_path="./LSTM_diagnostics/uc_coverage.png")
plot_hit_timeline(df_export, out_path="./LSTM_diagnostics/hit_timeline.png")

# （可選）把 UC 結果表也存起來，論文附錄很好用
uc_table.to_csv("./LSTM_diagnostics/uc_coverage_table.csv", index=False, encoding="utf-8-sig")
print("Saved: uc_coverage.png, hit_timeline.png, uc_coverage_table.csv")


# df_export 的 index 應該是 DATE
dfp = df_export.copy()
dfp.index = pd.to_datetime(dfp.index)

fig_path = f"./LSTM_diagnostics/{stem}_VaR_price{pd.to_datetime(TARGET_START_STR):%Y%m%d}_{pd.to_datetime(TARGET_END_STR):%Y%m%d}.png"  # 你原本存圖路徑變數

plt.figure(figsize=(12, 6))

plt.plot(dfp.index, dfp['price_true'], linewidth=0.9, alpha=0.9, label='Price (True)')

viol = dfp['viol_95'].fillna(0).astype(int).astype(bool)

ymin = float(dfp['price_true'].min())
ymax = float(dfp['price_true'].max())

plt.vlines(dfp.index[viol], ymin=ymin, ymax=ymax, color='red', alpha=0.25, linewidth=0.6,
           label='VaR 95% Violation Day')

plt.title(f"{ASSET_SYMBOL_ES1} Price with VaR 95% Violations (Vertical Lines)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
# # 1) 價格主線
# plt.plot(dfp.index, dfp['price_true'], linewidth=0.9, alpha=0.9, label='Price (True)')

# # 2) 95% 違規日（viol_95 已經在 df_export）
# viol = dfp['viol_95'].fillna(0).astype(int).astype(bool)

# # 3) 違規日上色（透明度高一點 -> alpha 大一點）
# shade_alpha = 0.35
# ax = plt.gca()

# for d in dfp.index[viol]:
#     ax.axvspan(d, d + pd.Timedelta(days=1), color='red', alpha=shade_alpha, linewidth=0)

# plt.title(f"{ASSET_SYMBOL_ES1} Price with VaR 95% Violations (Shaded)")
# plt.xlabel("Time")
# plt.ylabel("Price")
# plt.grid(True, alpha=0.3)
# plt.legend()

# plt.savefig(fig_path, dpi=300, bbox_inches='tight')
# plt.close()



✅ saved: ./LSTM_diagnostics/backtest_summary.csv
✅ saved: ./LSTM_diagnostics/transition_matrix.csv

--- backtest_summary ---
  level  alpha     n    x  viol_rate  expected_viol      LR_uc          p_uc  \
0    95   0.05  5072  326   0.064274         253.60  20.036085  7.599451e-06   
1    99   0.01  5072  102   0.020110          50.72  40.490601  1.975641e-10   

     LR_ind     p_ind      LR_cc          p_cc  pass_uc_5pct  pass_cc_5pct  
0  0.484693  0.486304  20.520778  3.499207e-05         False         False  
1  5.299958  0.021326  45.790560  1.139479e-10         False         False  

--- transition_matrix ---
  level  alpha     n    x  viol_rate   n00  n01  n10  n11      pi01      pi11  \
0    95   0.05  5072  326   0.064274  4443  302  302   24  0.063646  0.073620   
1    99   0.01  5072  102   0.020110  4873   96   96    6  0.019320  0.058824   

     pi_new  pi11_minus_pi01    LR_ind     p_ind  
0  0.073620         0.009974  0.484693  0.486304  
1  0.058824         0.039504  